# 윤서 — 국민체력100 동영상 API 비교

루틴 플레이어 작업 1. 7개 오퍼레이션의 명세와 실제 표본을 구분해 비교한다.
공식 명세: https://www.data.go.kr/data/15108846/openapi.do (확인: 2026-09-06).

금지 필드 `rptt_tcnt_nm`, `ecrg_cycl_nm`, `trng_hr_nm`은 분석·필터·시간 계산에 사용하지 않는다.
응답에서는 허용한 7개 필드와 영상 URL·파일명·제목·운동명·목록명만 남긴다.
키 없이도 명세 표까지 실행할 수 있다. 결과가 없는 것은 조회 결과 0건이라는 뜻이 아니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# 저장소 루트와 notebooks 디렉터리에서 모두 실행할 수 있다.
current_dir = Path.cwd().resolve()
project_root = next(
    path for path in (current_dir, *current_dir.parents)
    if (path / "src" / "nfa_video_api.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.nfa_video_api import OPERATIONS, VideoClient, collect_all, load_settings
from src.routine_comparison import (
    check_video_urls, combine_samples, compare_overlap, evaluate_scenario,
    profile_sample, save_comparison, schema_comparison,
)

display(pd.DataFrame(schema_comparison()))


## 동일 조건으로 표본 수집

저장소 `.env`의 `DATA_GO_KR_KEY` 또는 `API_KEY`를 사용한다. 키는 셀에 쓰거나 출력하지 않는다.
`DATA_DIR` 기본값은 `data/raw`다. 원본 데이터·키는 커밋하지 않는다.
기본은 오퍼레이션별 최대 3페이지, 페이지당 100행이며 시작·중간·끝을 고르게 조회한다.
최대 기본 요청 수는 21회, HTTP 일시 오류 재시도 포함 최대 63회다.
전수 비교가 필요하면 제공기관의 호출 제한을 확인한 뒤 `MAX_PAGES`를 늘린다.
표본은 무작위가 아니며 행은 영상 장면·메타정보 때문에 중복될 수 있다.


In [ ]:
PAGE_SIZE = 100
MAX_PAGES = 3
api_key, data_dir = load_settings(project_root)
samples = []
if api_key:
    client = VideoClient(api_key)
    samples = collect_all(client, page_size=PAGE_SIZE, max_pages=MAX_PAGES)
    del client
    print("수집 종료. 아래 표에서 실패·부분 수집 여부를 확인하세요.")
else:
    print("실데이터 미조회: .env 또는 환경변수에 인증키를 설정한 뒤 다시 실행하세요.")
del api_key


## 필드 품질·길이·중복 비교

`valid_rate`는 조회 행 기준 비율이다. `unique_videos`는 정확한 파일 URL 기준으로 중복을 제거한 수다.
숫자형 `vdo_len`은 공식 명세에 단위가 없어 기본적으로 시간 계산에서 제외한다.
실제 영상 길이와 대조해 단위를 확인한 오퍼레이션만 `NUMERIC_UNITS`에 `seconds` 또는 `minutes`로 설정한다.
`MM:SS`, `HH:MM:SS`, `분/초` 표기는 명시된 단위로 해석한다. 빈 값이나 0은 사용하지 않는다.


In [ ]:
NUMERIC_UNITS = {}  # 영상과 대조한 뒤 오퍼레이션 이름별 단위를 설정한다.
profiles = [profile_sample(sample, NUMERIC_UNITS.get(sample.operation)) for sample in samples]
overview = [{key: value for key, value in profile.items() if key != "fields"} for profile in profiles]
display(pd.DataFrame(overview))

field_rows = []
for profile in profiles:
    for name, values in profile["fields"].items():
        field_rows.append({"operation": profile["operation"], "field": name, **values})
display(pd.DataFrame(field_rows))
display(pd.DataFrame(compare_overlap(samples)))


## 조건별 준비 → 본 → 정리 구성

위의 실제 값 분포를 보고 `SCENARIOS`를 채운다. 모든 조건은 AND, 한 필드의 허용 값끼리는 OR다.
목적명을 임의로 매핑하지 않는다. 복합 분류값도 먼저 확인한 정확한 문자열을 전달한다.
필드가 없으면 통과시키지 않고 `unknown_rows`로 센다.
부위 필터는 사전에 검토한 허용 부위를 전달한다. 부위명만으로 통증 부위의 부담을 판정하지 않는다.
준비·본·정리마다 서로 다른 영상 1개를 골라 총길이를 맞출 수 있는지 확인한다.
이는 메타데이터 기반 구성 후보이며 운동 처방·강도 판단·실제 재생 보장은 아니다.

설정 형식:
```python
SCENARIOS = [{
    "name": "검토할 목적과 조건",
    "allowed_values": {"trng_aim_nm": {"응답에서 확인한 목적명"}},
    "stage_values": {
        "warmup": {"응답에서 확인한 준비 단계 값"},
        "main": {"응답에서 확인한 본 단계 값"},
        "cooldown": {"응답에서 확인한 정리 단계 값"},
    },
    "target_seconds": (600, 900),
}]
```
집 조건은 검토한 `tool_nm`, `trng_plc_nm` 값을 추가한다. 장소가 없는 오퍼레이션은 미확인으로 남긴다.
단순히 길이를 절반으로 줄이는 것을 저강도로 해석하지 않는다.


In [ ]:
SCENARIOS = []
# 단독 결과와 비교할 조합을 추가할 수 있다. 다른 행의 속성은 자동 결합하지 않는다.
OPERATION_GROUPS = [("TODZ_VDO_ROUTINE_I", "TODZ_VDO_TRNG_GUIDE_I")]
samples_by_name = {sample.operation: sample for sample in samples}
combined_samples = [
    combine_samples([samples_by_name[name] for name in group], NUMERIC_UNITS)
    for group in OPERATION_GROUPS if all(name in samples_by_name for name in group)
]
scenario_results = []
for scenario in SCENARIOS:
    for sample in [*samples, *combined_samples]:
        result = evaluate_scenario(
            sample, scenario["allowed_values"], scenario["stage_values"],
            scenario["target_seconds"], NUMERIC_UNITS.get(sample.operation),
        )
        scenario_results.append({"scenario": scenario["name"], "operation": sample.operation, **result})
if not SCENARIOS:
    print("조건별 구성 미검증: 실제 분류값을 확인한 뒤 SCENARIOS를 설정하세요.")
display(pd.DataFrame(scenario_results))


## URL 접근 확인과 최종 선별

`CHECK_URLS=True`로 바꾸면 오퍼레이션당 고유 URL 최대 5개를 HEAD 요청으로 확인한다.
전체 영상은 다운로드하지 않는다. HEAD 405는 재생 불가가 아니라 HEAD 미지원이다.
`video_header_accessible`도 브라우저 재생 성공은 아니다. 실제 플레이어에서 코덱·CORS·자동재생 정책을 별도로 확인한다.

명세상 주 사용 후보는 목적별루틴운동, 장소·체력요인 보완 후보는 운동처방가이드다.
아직 실데이터로 확정한 선정 결과는 아니다. 표본에서만 없는 영상을 전체에 없다고 결론내리지 않는다.
최소 조합은 주 후보에 다른 오퍼레이션이 더하는 고유 영상과 조건 충족 여부를 보고 정한다.
같은 URL이라도 장면별 조건이 다를 수 있어 여러 행의 속성을 임의로 합치지 않는다.


In [ ]:
CHECK_URLS = False
url_checks = []
if CHECK_URLS:
    for sample in samples:
        url_checks.extend({"operation": sample.operation, **check}
                          for check in check_video_urls(sample, limit=5))
display(pd.DataFrame(url_checks))

decisions = pd.DataFrame([
    {"operation": operation.name, "title": operation.title,
     "decision": "추가 확인 필요", "reason": "실데이터·조건별 구성·재생 검증 후 작성"}
    for operation in OPERATIONS
])
display(decisions)


## 로컬 결과 저장

수집한 허용 필드 표본과 비교표를 `DATA_DIR/routine_player/comparison.json`에 저장한다.
시나리오와 URL 접근 검사, 담당자 선정표는 같은 디렉터리의 `evaluation.json`에 저장한다.
기본 경로는 Git 제외 대상이다. `DATA_DIR`을 바꾸면 저장 파일의 Git 제외 여부도 확인한다.


In [ ]:
import json

if samples:
    output_path = save_comparison(samples, data_dir, NUMERIC_UNITS)
    evaluation = {"scenarios": scenario_results, "url_checks": url_checks,
                  "decisions": decisions.to_dict(orient="records")}
    evaluation_path = output_path.with_name("evaluation.json")
    evaluation_path.write_text(
        json.dumps(evaluation, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8", newline="\n",
    )
    print("비교 결과 저장:", output_path)
else:
    print("실데이터 미조회 상태이므로 결과 파일을 생성하지 않았습니다.")
